In [82]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd
import os
import re
from string import punctuation
from datetime import datetime
import json

#DIRNAME = 'jeopardy/'

In [10]:
#jeo = os.listdir(DIRNAME)

In [17]:
#jeot = open(DIRNAME + jeo).read()

In [15]:
with open("jeopardy.json", "r") as file:
    jeo = json.load(file)
#Used a geeksforgeeks tutorial to figure this out (Not AI)

In [18]:
df = pd.DataFrame(jeo)

In [64]:
#df

In [65]:
df.value = pd.to_numeric(df.value.astype(str).str.replace(r"[$,]", "", regex = True), errors = "coerce")
df = df.dropna(subset = ["question", "value"]).copy()

In [66]:
cutoff = 1000

df["label"] = (df["value"] >= cutoff).map({
    True: "high value",
    False: "low value"
})


In [57]:
df

,category,air_date,question,value,answer,round,show_number,label
0,HISTORY,2004-12-31,"'For the last 8 years of his life, Galileo was...",200.0,Copernicus,Jeopardy!,4680,low value
1,ESPN's TOP 10 ALL-TIME ATHLETES,2004-12-31,'No. 2: 1912 Olympian; football star at Carlis...,200.0,Jim Thorpe,Jeopardy!,4680,low value
2,EVERYBODY TALKS ABOUT IT...,2004-12-31,'The city of Yuma in this state has a record a...,200.0,Arizona,Jeopardy!,4680,low value
3,THE COMPANY LINE,2004-12-31,"'In 1963, live on ""The Art Linkletter Show"", t...",200.0,McDonald\'s,Jeopardy!,4680,low value
4,EPITAPHS & TRIBUTES,2004-12-31,"'Signer of the Dec. of Indep., framer of the C...",200.0,John Adams,Jeopardy!,4680,low value
...,...,...,...,...,...,...,...,...
216924,OFF-BROADWAY,2006-05-11,'In 2006 the cast of this long-running hit emb...,2000.0,Stomp,Double Jeopardy!,4999,high value
216925,RIDDLE ME THIS,2006-05-11,'This Puccini opera turns on the solution to 3...,2000.0,Turandot,Double Jeopardy!,4999,high value
216926,"""T"" BIRDS",2006-05-11,'In North America this term is properly applie...,2000.0,a titmouse,Double Jeopardy!,4999,high value
216927,AUTHORS IN THEIR YOUTH,2006-05-11,"'In Penny Lane, where this ""Hellraiser"" grew u...",2000.0,Clive Barker,Double Jeopardy!,4999,high value


In [58]:
#Start of the Naive Bayes Portion

In [59]:
x = df["question"]
y = df["label"]

In [60]:
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state = 1)


In [61]:
tfidf_vectorizer = TfidfVectorizer(use_idf = True)
x_train_tf = tfidf_vectorizer.fit_transform(x_train)
x_test_tf = tfidf_vectorizer.transform(x_test)

In [62]:
naive_bayes = MultinomialNB()
naive_bayes.fit(x_train_tf, y_train)
predictions = naive_bayes.predict(x_test_tf)

In [63]:
print("Naive Bayes Accuracy: ", accuracy_score(y_test, predictions))

Accuracy:  0.714462530942915


In [70]:
#Support Vector Machines

In [79]:
svm_model = make_pipeline(TfidfVectorizer(use_idf=True), LinearSVC())
svm_model.fit(x_train, y_train)
svm_predictions = svm_model.predict(x_test)
svm_accuracy = accuracy_score(y_test, svm_predictions)
print("Support Vector Machines accuracy:", svm_accuracy)

Support Vector Machines' accuracy: 0.6908333958442727


In [80]:
#Logistic Regression

In [81]:
lr_model = make_pipeline(
    TfidfVectorizer(use_idf=True),
    LogisticRegression(max_iter=1000)
)

lr_model.fit(x_train, y_train)
lr_predictions = lr_model.predict(x_test)
lr_accuracy = accuracy_score(y_test, lr_predictions)
print("Logistic Regression accuracy:", lr_accuracy)

Logistic Regression accuracy: 0.7106368614507539
